# 6.1 정규화: \(\lambda\)를 숫자로 다루다 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml1/chapter06_1_regularization.ipynb)

책 본문: [6.1 정규화: 편향-분산 복습과 L1/L2 페널티](https://smhanlab.com/book-ml/kor/ml1/chapter06/1.html)

이 노트북은 책 6.1절의 핵심 주장들을 **모두 코드로 검증**합니다:

1. Ridge 정규방정식 \(w^*=(X^TX+\lambda D)^{-1}X^Ty\)가 본문 "손으로 한 번"의 3개 점(\(\lambda=0,2,10\))을 재현하는지
2. L1(Lasso)이 잡음 특징을 **정확히 0**으로 만들고 L2(Ridge)는 그렇지 않은지
3. Lasso **계수 경로**: \(\lambda\)를 0에서 키우면 잡음 계수는 \(\lambda{\approx}0.13\)에서 *스냅*처럼 0이 되고, 진짜 계수는 서서히 줄다(그림 1)
4. L1이 "왜" 정확히 0이 되는지 — 좌표하강 + soft-thresholding을 손으로 구현해 sklearn과 대조
5. 제약 영역의 기하학: L2의 **원** vs L1의 **마름모**가 등고선에서 다른 지점에서 만나는지(그림 2)
6. \(\lambda\)가 4.1절의 **편향-분산 트레이드오프를 실제로 조절하는 손잡이**인지 — 15차 다항식 모델에서 bootstrap으로 \(\text{Bias}^2\)와 \(\text{Var}\)를 직접 측정(그림 3)


## 1. Ridge 정규방정식: 본문 "손으로 한 번"의 3개 점

본문은 데이터 \(X=[1,2,3]\), \(y=[3,5,7]\)(\(\lambda=0\)의 해 \(w^*=[1,2]\))에 \(\lambda=2,10\)을 대입해 \(w^*=[3,1]\), \(w^*\approx[4.33,0.33]\)이 나온다고 손으로 계산했다(\(D\)는 bias를 제외한 대각행렬). 같은 식을 닫힌 형태로 코딩해 확인한다. bias(첫 번째 좌표)는 **정규화하지 않음**(\(D=\text{diag}(0,1)\)).


In [1]:
import numpy as np

Xb = np.array([[1.0],[2.0],[3.0]])      # 특징 x
X1 = np.array([[1.0],[1.0],[1.0]])      # bias 열
Mx = np.column_stack([X1, Xb])          # 열 순서 [bias, x]
y  = np.array([3.0, 5.0, 7.0])
D  = np.array([[0.0, 0.0],[0.0, 1.0]])  # x(기울기)만 정규화

for lam in [0.0, 2.0, 10.0]:
    w = np.linalg.solve(Mx.T @ Mx + lam * D, Mx.T @ y)
    print(f"lambda={lam:5.1f}:  w0(bias)={w[0]:.4f}   w1(slope)={w[1]:.4f}")

# 본문 "자주 하는 실수": bias까지 정규화(틀린 D=I)
w_wrong = np.linalg.solve(Mx.T @ Mx + 2.0 * np.eye(2), Mx.T @ y)
print(f"틀린 경우(lambda=2, bias도 정규화): w0={w_wrong[0]:.4f}  w1={w_wrong[1]:.4f}")

# 본문 값과 대조
assert abs(np.linalg.solve(Mx.T@Mx, Mx.T@y)[0] - 1.0) < 1e-9
assert abs(np.linalg.solve(Mx.T@Mx + 2*D, Mx.T@y)[0] - 3.0) < 1e-9 and abs(np.linalg.solve(Mx.T@Mx + 2*D, Mx.T@y)[1] - 1.0) < 1e-9
assert abs(np.linalg.solve(Mx.T@Mx + 10*D, Mx.T@y)[1] - 0.3333) < 5e-4
print("\n본문 '손으로 한 번'의 세 점([1,2],[3,1],[4.33,0.33])을 코드가 정확히 재현했다.")

lambda=  0.0:  w0(bias)=1.0000   w1(slope)=2.0000
lambda=  2.0:  w0(bias)=3.0000   w1(slope)=1.0000
lambda= 10.0:  w0(bias)=4.3333   w1(slope)=0.3333
틀린 경우(lambda=2, bias도 정규화): w0=0.8182  w1=1.8182

본문 '손으로 한 번'의 세 점([1,2],[3,1],[4.33,0.33])을 코드가 정확히 재현했다.


## 2. "실습" 예: 잡음 특징은 L1이 정확히 0, L2는 아님

본문의 예: \(x_1\)은 \(y=2x_1+1\)을 만드는 진짜 특징, \(x_2\)는 \(y\)와 무관한 잡음. \(\alpha\)를 키우면 Ridge의 \(x_2\) 계수는 작아지지만 **0이 안 되고**, Lasso는 **정확히 0**이 된다.

In [2]:
from sklearn.linear_model import Ridge, Lasso
import warnings
warnings.filterwarnings("ignore")

x1 = np.array([1,2,3,4,5,6], dtype=float)
x2 = np.array([5,3,8,1,9,2], dtype=float)   # y와 무관한 잡음
y  = 2 * x1 + 1
X  = np.column_stack([x1, x2])

for alpha in [0.1, 1.0, 5.0]:
    r = Ridge(alpha=alpha).fit(X, y)
    l = Lasso(alpha=alpha).fit(X, y)
    print(f"alpha={alpha}:  ridge x2={r.coef_[1]:.4f} (0 아님)   "
          f"lasso x2={l.coef_[1]:.4f} (정확히 0={l.coef_[1]==0.0})")

alpha=0.1:  ridge x2=-0.0004 (0 아님)   lasso x2=-0.0000 (정확히 0=True)
alpha=1.0:  ridge x2=-0.0040 (0 아님)   lasso x2=-0.0000 (정확히 0=True)
alpha=5.0:  ridge x2=-0.0153 (0 아님)   lasso x2=0.0000 (정확히 0=True)


## 3. Lasso 계수 경로: 0은 "점진적"이 아니라 "도착지"

본문 §계수경로의 실험. 30개 점에서 \(x_1\sim U(-1,1)\), \(y=3x_1+\)잡음(\(\sigma=1\)), \(x_2\sim N(0,1)\)(순수 잡음 특징)을 만들고, \(\lambda\)를 \(0.01\)에서 \(2.5\)까지 훑으며 두 계수를 기록한다. **진짜 특징** \(x_1\)의 계수는 서서히 줄고, **잡음 특징** \(x_2\)의 계수는 \(\lambda\)가 임계값(\(\approx0.13\))을 넘는 순간 정확히 0이 되어 *다시 한 번도* 0에서 벗어나지 않는다 — soft-thresholding(\(|z_j|\le\lambda\)면 0)의 직접 모습.


In [3]:
import numpy as np, warnings
warnings.filterwarnings("ignore")
from sklearn.linear_model import Lasso

rng = np.random.default_rng(0)
m = 30
x1 = rng.uniform(-1, 1, m)
x2 = rng.normal(0, 1, m)          # y와 무관한 잡음 특징
y  = 3 * x1 + rng.normal(0, 1, m)
X  = np.column_stack([x1, x2])

als = np.linspace(0.001, 2.5, 800)
p1, p2 = [], []
for a in als:
    l = Lasso(alpha=a, max_iter=20000).fit(X, y)
    p1.append(l.coef_[0]); p2.append(l.coef_[1])
p1, p2 = np.array(p1), np.array(p2)

# x2: 0이 아닌 값 -> 정확히 0으로 전환되는 첫 지점
t = None
for i in range(1, len(als)):
    if p2[i] == 0.0 and p2[i - 1] != 0.0:
        t = i
        break
print(f"OLS(lambda->0): x1={p1[0]:.3f}, x2={p2[0]:.3f}")
print(f"x2(잡음)가 정확히 0이 되는 순간: alpha={als[t]:.4f} (직전 값 {p2[t-1]:.4f})")
print(f"x2가 정확히 0인 시그먼트: {np.sum(p2==0.0)}/{len(als)}  (0이 된 뒤 다시 벗어남: {np.any(p2[t:]!=0.0)})")
print(f"x1(진짜)는 alpha={als[np.argmax(p1==0.0)]:.3f}에서야 정확히 0")

import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.plot(als, p1, "-o", ms=3, lw=1.6, color="tab:blue", label="True feature $x_1$ (true coefficient 3)")
ax.plot(als, p2, "-s", ms=3, lw=1.6, color="tab:red",  label="Noise feature $x_2$ (irrelevant)")
ax.axvline(als[t], color="red", ls="--", lw=1, alpha=0.6)
ax.text(als[t] * 1.15, 0.35, f"$\\lambda\\approx{als[t]:.2f}$ (snaps to 0 here)",
        fontsize=9, color="red")
ax.axhline(0, color="k", lw=0.8)
ax.set_xlabel("$\\lambda$ (Lasso alpha)"); ax.set_ylabel("Coefficient")
ax.set_title("Lasso coefficient path \u2014 the noise coefficient drops to exactly 0 the moment \u03b1 passes the threshold, while the true coefficient shrinks gradually")
ax.legend(fontsize=9); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch06_1_lasso_path.svg")
print("figure saved -> kor/src/images/ch06_1_lasso_path.svg")


OLS(lambda->0): x1=3.089, x2=0.149
x2(잡음)가 정확히 0이 되는 순간: alpha=0.1324 (직전 값 0.0010)
x2가 정확히 0인 시그먼트: 758/800  (0이 된 뒤 다시 벗어남: False)
x1(진짜)는 alpha=1.224에서야 정확히 0


figure saved -> kor/src/images/ch06_1_lasso_path.svg


## 4. L1이 "왜" 정확히 0: 좌표하강 + soft-thresholding

Lasso의 목적함수는 \(|w_j|\) 때문에 미분불가능하다. 표준 해법은 **좌표하강**(coordinate descent) — 한 번에 \(w_j\) 하나만 놓고 나머지를 고정하면 목적함수가 \(w_j\)의 이차함수가 되고, 그 최소점은 **soft-thresholding**

\[w_j \leftarrow \frac{\mathcal{T}_\lambda(z_j)}{\sum_i x_{ij}^2 / m},
\qquad
\mathcal{T}_\lambda(z) = \text{sign}(z)\,\max(|z|-\lambda,\, 0)\]

이다. \(|z_j|\le\lambda\)면 \(w_j\)는 **0으로 끌려간다** — 본문 기하학적 직관(마름모 꼭짓점)의 대수적 모습. 이 함수를 구현해 본문 §2의 데이터에서 \(x_2\) 계수가 0이 되는 것을 직접 보고, sklearn의 Lasso와 같은 해인지 대조한다.

In [4]:
def soft(z, lam):
    return float(np.sign(z) * max(abs(z) - lam, 0.0))

# bias는 정규화에서 제외(sklearn Lasso의 fit_intercept=True와 동일):
#   bias = (y - Xf w).mean()  (OLS 갱신), 특징만 soft-thresholding
def lasso_coordinate_descent(Xf, y, lam, max_pass=100000, tol=1e-13):
    m = len(y); w = np.zeros(Xf.shape[1]); b = 0.0
    col_sq = (Xf ** 2).sum(0) / m
    for _ in range(max_pass):
        b = (y - Xf @ w).mean()                     # bias: 미분가능, OLS
        for j in range(Xf.shape[1]):
            r = y - b - Xf @ w + w[j] * Xf[:, j]    # j번째 제외 잔차
            z = float(Xf[:, j] @ r) / m
            w[j] = soft(z, lam) / col_sq[j]         # 특징: soft-threshold
    return w, b

Xf = np.column_stack([x1, x2])   # 특징만(bias 제외), sklearn Lasso와 동일한 설정
for alpha in [0.5, 2.0, 5.0]:
    wcd, bcd = lasso_coordinate_descent(Xf, y, alpha)
    sk = Lasso(alpha=alpha).fit(Xf, y)
    ok = np.allclose(wcd, sk.coef_, atol=1e-6) and abs(bcd - sk.intercept_) < 1e-6
    print(f"alpha={alpha}: 좌표하강 x1={wcd[0]:.4f} x2={wcd[1]:.4f} bias={bcd:.4f}   "
          f"sklearn x1={sk.coef_[0]:.4f} x2={sk.coef_[1]:.4f} bias={sk.intercept_:.4f}   일치={ok}")
    print(f"          -> 잡음 특징 x2 계수 = {wcd[1]:.6f}  (정확히 0: {wcd[1]==0.0})")

# 모든 alpha에서 x2가 정확히 0이 되는 것이 soft-thresholding의 직접 결과
assert all(lasso_coordinate_descent(Xf, y, a)[0][1] == 0.0 for a in [0.5, 2.0, 5.0])
print("bias를 정규화에서 제외한 좌표하강이 sklearn Lasso와 완전히 일치하며, 잡음 특징 x2는 정확히 0으로 소거된다.")

alpha=0.5: 좌표하강 x1=1.8445 x2=0.0000 bias=0.2540   sklearn x1=1.8445 x2=0.0000 bias=0.2540   일치=True
          -> 잡음 특징 x2 계수 = 0.000000  (정확히 0: True)


alpha=2.0: 좌표하강 x1=0.0000 x2=0.0000 bias=0.3808   sklearn x1=0.0000 x2=0.0000 bias=0.3808   일치=True
          -> 잡음 특징 x2 계수 = 0.000000  (정확히 0: True)


alpha=5.0: 좌표하강 x1=0.0000 x2=0.0000 bias=0.3808   sklearn x1=0.0000 x2=0.0000 bias=0.3808   일치=True
          -> 잡음 특징 x2 계수 = 0.000000  (정확히 0: True)


bias를 정규화에서 제외한 좌표하강이 sklearn Lasso와 완전히 일치하며, 잡음 특징 x2는 정확히 0으로 소거된다.


## 5. 제약 영역의 기하학: 원(L2) vs 마름모(L1)  [그림 2]

손실 \(J(w)\)의 등고선(타원) 위에 제약 영역을 겹쳐 본다. L2 제약 \(\|w\|_2\le t\)는 **원**(원점 중심), L1 제약 \(\|w\|_1\le t\)는 **마름모**(축 위의 꼭짓점 \((\pm t,0),(0,\pm t)\)). "약한 제약"(작은 \(t\))일수록 등고선이 제약 경계에 먼저 닿는 지점 — L1은 **축 위의 꼭짓점**(어떤 좌표가 정확히 0)에서, L2는 **두 좌표 모두 0이 아닌 점**에서 닿는 것이 일반적인데, 아래에서 실제 닿는 지점을 계산해 확인한다. 데이터는 본문 "기하학적 직관" 단락의 \(X=[1,2,3,4]\), \(y=[3,5,7,9]\)(OLS 해가 정확히 \((1,2)\)인 완전 직선). \(t=2\)이면 두 제약 영역 모두 OLS 해 안쪽이라 최적점은 *제약 경계 위*에서 결정된다.


In [5]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

x = np.array([1.,2.,3.,4.]); yv = np.array([3.,5.,7.,9.])
Xd = np.column_stack([np.ones(4), x])
def sse(w):
    return float(np.sum((yv - Xd @ w)**2))
wols = np.linalg.lstsq(Xd, yv, rcond=None)[0]   # OLS (정규화 없음) = (1,2)

w0g = np.linspace(-3,3,400); w1g = np.linspace(-3,3,400)
W0,W1 = np.meshgrid(w0g, w1g)
Wm = np.stack([W0.ravel(), W1.ravel()], axis=1)
SSE = ((Wm @ Xd.T - yv)**2).sum(1).reshape(W0.shape)

t = 2.0
th = np.linspace(0, 2*np.pi, 400)
wl2 = np.stack([t*np.cos(th), t*np.sin(th)], axis=1)
s2 = ((wl2 @ Xd.T - yv)**2).sum(1)
wL2 = wl2[int(np.argmin(s2))]
verts = np.array([[t,0.],[0.,t],[-t,0.],[0.,-t]])
segs = [(verts[k], verts[(k+1)%4]) for k in range(4)]
perim = np.vstack([(a[:,None] + (b-a)[:,None]*np.linspace(0,1,2000)[None,:]).T for a,b in segs])
s1 = ((perim @ Xd.T - yv)**2).sum(1)
wL1 = perim[int(np.argmin(s1)), :]

fig, ax = plt.subplots(figsize=(8,7))
cs = ax.contour(W0, W1, SSE, levels=9, cmap="viridis", alpha=0.85)
ax.clabel(cs, fmt="%.0f", fontsize=8)
ax.plot(t*np.cos(th), t*np.sin(th), "b-", lw=2.2, label=f"L2: $|w|_2 \leq {t}$ (circle)")
ax.plot([verts[0][0],verts[1][0],verts[2][0],verts[3][0],verts[0][0]],
        [verts[0][1],verts[1][1],verts[2][1],verts[3][1],verts[0][1]],
        "r-", lw=2.2, label=f"L1: $|w|_1 \leq {t}$ (diamond)")
ax.plot(*wols, "k*", ms=15, label=f"OLS = ({wols[0]:.2f}, {wols[1]:.2f})")
ax.plot(*wL2, "bo", ms=8, label=f"L2 optimum ({wL2[0]:.2f},{wL2[1]:.2f}), SSE={sse(wL2):.2f}")
ax.plot(*wL1, "rs", ms=8, label=f"L1 optimum ({wL1[0]:.2f},{wL1[1]:.2f}), SSE={sse(wL1):.2f}")
ax.plot(0,0,"k+",ms=10)
ax.set_xlabel("$w_0$ (bias)"); ax.set_ylabel("$w_1$ (slope)")
ax.set_title("Loss contours vs. constraint sets (t=2) \u2014 the L2 contour meets the circle boundary, the L1 contour meets a diamond vertex (on an axis)")
ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=0.3); ax.set_aspect("equal")
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch06_1_l1_l2_geometry.svg")
print(f"t={t}:  L2 최적점=({wL2[0]:.3f},{wL2[1]:.3f}) SSE={sse(wL2):.2f}")
print(f"        L1 최적점=({wL1[0]:.3f},{wL1[1]:.3f}) SSE={sse(wL1):.2f}  <- w_0이 정확히 0 (축 위의 꼭짓점)")
print("figure saved -> kor/src/images/ch06_1_l1_l2_geometry.svg")


t=2.0:  L2 최적점=(0.672,1.884) SSE=1.60
        L1 최적점=(0.000,2.000) SSE=4.00  <- w_0이 정확히 0 (축 위의 꼭짓점)
figure saved -> kor/src/images/ch06_1_l1_l2_geometry.svg


## 6. \(\lambda\)가 편향-분산을 조절하는 손잡이  [그림 3]

4.1절의 분해 \(\mathbb{E}[(y-\hat f)^2] = \text{Bias}^2 + \text{Var} + \sigma^2\)을 **선형모델의 \(\lambda\) 축**으로 다시 측량한다. \(f^*(x)=\sin 2\pi x\)에 잡음 \(\sigma=0.5\)를 얹은 100점으로, 모델은 **15차 다항식 특징**(\(\lambda=0\)이면 100점을 거의 자유자재로 통과할 만큼 유연)이며, 70/30 재추출을 300번 반복해 각 \(\lambda\)마다 40개 test 점에서 \(\text{Bias}^2\), \(\text{Var}\), test RMSE를 계산한다. \(\lambda=0\)에서 Var가 극단적으로 크고(과적합), \(\lambda\)를 올리면 **Var가 급감하고 Bias\(^2\)가 느리게 상승**하며, RMSE 곡선은 **U자형**이 된다(바닥 \(\approx\lambda=0.001\)). 노이즈 플로어(\(\lambda\to\infty\), 평균값만 예측)까지의 경계가 오른쪽 그림의 회색 점선이다.


In [6]:
import numpy as np, warnings
warnings.filterwarnings("ignore")
rng = np.random.default_rng(42)
fstar = lambda x: np.sin(2*np.pi*x)
xs = rng.uniform(-1,1,100); ys = fstar(xs) + rng.normal(0,0.5,100)
xt = np.linspace(-1,1,40); yt = fstar(xt) + rng.normal(0,0.5,40)
def design(x): return np.column_stack([x**j for j in range(1,16)])  # 15차 다항식 특징(bias 열 없음)
def fit_ridge(F, yv, a):
    m=len(yv); f0,y0 = F.mean(0), yv.mean(); Fc,yc = F-f0, yv-y0
    w = np.linalg.solve(Fc.T@Fc + m*a*np.eye(Fc.shape[1]), Fc.T@yc)
    return w, y0 - w@f0   # (feature weights, bias)
def bv(Ftr,ytr,Fte,yte,fte,a,nres=300,seed=42):
    r = np.random.default_rng(seed); m=len(ytr); P=[]
    for s in range(nres):
        idx = r.choice(m, size=int(0.7*m), replace=False)
        w,b = fit_ridge(Ftr[idx], ytr[idx], a); P.append(Fte@w + b)
    P = np.array(P); fb = P.mean(0)
    return ((fb-fte)**2).mean(), ((P-fb[None])**2).mean(0).mean(), ((P-yte[None])**2).mean()
Ftr, ytr = design(xs), ys; Fte, yte = design(xt), yt; fte = fstar(xt)
grid = [0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
print(f"{'alpha':>8} {'Bias^2':>8} {'Var':>10} {'testRMSE':>9}")
for a in grid:
    b2,v,mse = bv(Ftr,ytr,Fte,yte,fte,a)
    print(f"{a:8.3f} {b2:8.4f} {v:10.4f} {np.sqrt(mse):9.4f}")
print(f"\n노이즈 플로어 RMSE(=y의 표준편차, w=0 극한) = {yt.std():.4f}")
print("-> lambda=0: Var가 극단적(과적합), lambda=0.001: U자 바닥, lambda=100: 노이즈 플로어에 붙음(과소적합)")


   alpha   Bias^2        Var  testRMSE
   0.000   5.6392   110.8977   10.8099
   0.001   0.2404     0.0301    0.6643
   0.010   0.2555     0.0153    0.6834
   0.100   0.4001     0.0077    0.8044
   1.000   0.4378     0.0039    0.8471
  10.000   0.4797     0.0038    0.8800
 100.000   0.4880     0.0038    0.8860

노이즈 플로어 RMSE(=y의 표준편차, w=0 극한) = 0.8804
-> lambda=0: Var가 극단적(과적합), lambda=0.001: U자 바닥, lambda=100: 노이즈 플로어에 붙음(과소적합)


In [7]:
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
res = [bv(Ftr,ytr,Fte,yte,fte,a) for a in grid]
B2 = np.array([r[0] for r in res]); VV = np.array([r[1] for r in res]); RM = np.array([np.sqrt(r[2]) for r in res])
fig, (a1,a2) = plt.subplots(1,2, figsize=(12,4.5))
a1.semilogx(grid, B2, "-o", label="Bias$^2$")
a1.semilogx(grid, VV, "-s", label="Variance")
a1.set_xlabel("$\\lambda$ (log scale)"); a1.set_ylabel("Error component")
a1.set_title("Degree-15 polynomial model: raising $\\lambda$ cuts variance sharply while bias grows slowly")
a1.legend(); a1.grid(alpha=0.3)
a2.semilogx(grid, RM, "-o", label="test RMSE")
a2.axhline(yt.std(), color="gray", ls="--", lw=1.5, label=f"Noise floor RMSE={yt.std():.3f}")
a2.set_xlabel("$\\lambda$ (log scale)"); a2.set_ylabel("test RMSE")
a2.set_title("Validation performance curve \u2014 U-shaped (overfitting $\\lambda\\to0$ \u2194 underfitting $\\lambda\\to\\infty$)")
a2.legend(); a2.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("/home/smhan/book-ml/kor/src/images/ch06_1_lambda_biasvar.svg")
print("figure saved -> kor/src/images/ch06_1_lambda_biasvar.svg")


figure saved -> kor/src/images/ch06_1_lambda_biasvar.svg


## 정리: 다음으로

- **6.2 교차검증**: §6의 U자형 \(\lambda\) 곡선에서 "바닥(최적점)"을 *고르는 것*은 검증 데이터가 필요하므로, 6.2절에서 k-fold CV로 \(\lambda\)를 데이터가 정하게 한다.
- **6.3 train/val/test**: \(\lambda\)를 val로 고르고 test로 한 번만 확인하는 절차 — 이 절의 U자형 곡선을 *정직하게* 읽는 방법(선택 편향 없이).
